# Image Semantic Demo — Spatial Exploration and Semantic Categorization

This notebook demonstrates the broadest part of the `tobii-pytracker` analysis API on real 12-image sessions. It combines behavioral accuracy with fixation, saccade, scanpath, spatial-entropy, heatmap/focus-map, clustering, and image-grid AOI analyses.

The primary scientific question is whether spatial exploration differs across **BIOLOGICAL**, **OBJECT**, and **SCENE** categorization. The 3×3 grid is a coarse spatial AOI model, not a semantic object annotation.

## Scientific context and experimental logic

This notebook follows the design described in [Image Semantic Demo](../../../docs/basic_examples/image_semantic_demo.md). The task is a forced-choice semantic categorization experiment intended to demonstrate how behavioral classification and spatial gaze analysis can be combined for image stimuli.

### Experimental structure

- 12 distinct images;
- 4 `BIOLOGICAL`, 4 `OBJECT`, and 4 `SCENE` stimuli;
- only three response buttons are presented: `BIOLOGICAL`, `OBJECT`, and `SCENE`;
- the generic uncertainty response is deliberately absent;
- categories are mixed using a constrained order: adjacent trials cannot share a category, and each consecutive group of three contains one image from every category;
- every image exposes a coarse 3×3 spatial grid through `image_bboxes`.

This ordering reduces simple response repetition and category-block expectation while preserving local balance.

### Scientific model

The documentation frames the experiment around the distinction between **localized evidence sampling** and **broad spatial exploration**. A compact object may be categorized from a relatively restricted region, whereas scene categorization can require more distributed visual sampling. The 3×3 grid therefore acts as a deliberately simple spatial AOI system rather than a semantic object detector.

| Documentation hypothesis | Operational measure in this notebook | Primary interpretation |
|---|---|---|
| H1 — Category effect | fixation, saccade, entropy, grid-attention summaries by category | Do gaze patterns differ across semantic classes? |
| H2 — Spatial breadth | visited grid cells, convex hull, entropy, scanpath distance | Are SCENE trials explored more broadly than compact OBJECT trials? |
| H3 — Decision efficiency | accuracy vs trial duration/fixation/saccade metrics | Are correct decisions reached with different visual effort? |
| H4 — Stimulus heterogeneity | per-image diagnostics | Do individual exemplars deviate from category averages? |

### Why this notebook uses the broadest analyzer set

This demo is the best match for the public image-analysis API in `tobii-pytracker`. The stored `image_bboxes` can be consumed directly by `BBoxAttentionAnalyzer`, while fixation, saccade, scanpath, entropy, heatmap, focus-map, and clustering tools provide complementary views of the same trial. The notebook uses these components to demonstrate the library rather than reproducing them locally.

### Measurement caution

The 3×3 grid is a **coarse spatial partition**, not a semantic annotation of objects. A central-cell dwell effect should therefore be described spatially, not interpreted as attention to a specific object unless separate semantic AOIs are available. Similarly, heatmaps and entropy summarize distribution, but they do not by themselves identify why a region was informative.

Because there are only four heterogeneous exemplars per category, stimulus-level inspection is essential. Strong category-general claims would require a larger and more tightly controlled stimulus set.

The demo documentation cites Duchowski (2017) and Goldberg & Kotval (1999) as methodological background for eye tracking and interface evaluation.


## 1. Analysis parameters

This cell makes all event- and visualization-sensitive settings explicit. Fixation and saccade thresholds affect event counts, entropy depends on the gaze distribution supplied to the analyzer, and DBSCAN clustering is particularly parameter-sensitive.

Treat the clustering parameters as exploratory. They should not be tuned to manufacture visually attractive clusters or category differences.


In [ ]:
SELECTED_SESSION = None
FIXATION_PARAMS = {"method":"dispersion", "dispersion_threshold":50.0, "min_duration":0.10}
SACCADE_PARAMS = {"method":"ivt", "velocity_threshold":100.0, "min_duration":0.01, "filter_micro_saccades":False}
SPATIAL_ENTROPY_BINS = 40
CLUSTER_EPS = 75.0
CLUSTER_MIN_SAMPLES = 5
RUN_FIXATION_SENSITIVITY = False
SENSITIVITY_DISPERSION_THRESHOLDS = [40.0, 50.0, 60.0]

REPRESENTATIVE_CATEGORY = None  # optionally choose "biological", "object", or "scene"


## 2. Load a real image session

The notebook analyzes collected image trials only. `DataLoader` preserves the connection between behavioral responses, gaze samples, screenshots, and the stored 3×3 image-grid geometry.

The most recent valid session is selected for convenience; explicit session selection is preferable when producing a reproducible report or comparing participants.

> **tobii-pytracker support:** `CustomConfig` and `DataLoader` are used to load the session and preserve the link between behavioral responses, gaze samples, screenshots, and image geometry.


In [ ]:
from pathlib import Path
import ast
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tobii_pytracker.configs.custom_config import CustomConfig
from tobii_pytracker.analyze import (
    DataLoader,
    FixationAnalyzer,
    ScanpathsAnalyzer,
)


def find_demo_root() -> Path:
    """Locate tobii-pytracker-demo from common Jupyter launch locations."""
    start = Path.cwd().resolve()
    candidates = [start, start / "tobii-pytracker-demo", *start.parents]
    for candidate in candidates:
        if (candidate / "examples").is_dir() and (candidate / "output").is_dir():
            return candidate
        nested = candidate / "tobii-pytracker-demo"
        if (nested / "examples").is_dir() and (nested / "output").is_dir():
            return nested
    raise FileNotFoundError("Could not locate the tobii-pytracker-demo repository root.")


def newest_subject(loader: DataLoader) -> str:
    subjects = loader.get_subjects()
    if not subjects:
        raise FileNotFoundError(f"No experiment sessions found under {loader.output_root}")
    def mtime(subject: str) -> float:
        return (loader.output_root / subject / "data.csv").stat().st_mtime
    return max(subjects, key=mtime)


def prepare_session(config_path: Path, subject: str | None = None):
    """Load one real session with DataLoader; never synthesize replacement data."""
    config = CustomConfig(str(config_path))
    loader = DataLoader(config=config, root=DEMO_ROOT)
    selected = subject or newest_subject(loader)
    raw = loader.get_subject_data(selected, flatten=False).reset_index(drop=True)
    raw.insert(0, "set_name", selected)
    raw["slide_index"] = np.arange(len(raw), dtype=int)
    flat = loader.get_subject_data(selected, flatten=True)
    return loader, selected, raw, flat


def safe_parse(value, expected_type, default):
    if isinstance(value, expected_type):
        return value
    text = "" if value is None else str(value).strip()
    if not text or text.lower() == "nan":
        return default
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, expected_type):
                return parsed
        except Exception:
            pass
    return default


def point_in_centered_bbox(x: float, y: float, bbox: dict, margin: float = 2.0) -> bool:
    try:
        cx, cy, w, h = (float(bbox[k]) for k in ("cx", "cy", "w", "h"))
    except (KeyError, TypeError, ValueError):
        return False
    return (cx - w/2 - margin <= x <= cx + w/2 + margin and
            cy - h/2 - margin <= y <= cy + h/2 + margin)


def normalize_token(value) -> str:
    return re.sub(r"[^0-9a-ząćęłńóśźż]+", "", str(value).casefold())


def trial_gaze_counts(flat: pd.DataFrame, n_trials: int) -> pd.Series:
    counts = pd.Series(0, index=range(n_trials), dtype=int)
    if not flat.empty and {"slide_index", "avg_gaze_x"}.issubset(flat.columns):
        observed = flat.dropna(subset=["avg_gaze_x", "avg_gaze_y"]).groupby("slide_index").size()
        for idx, count in observed.items():
            if int(idx) in counts.index:
                counts.loc[int(idx)] = int(count)
    return counts


def trial_observed_duration(flat: pd.DataFrame, n_trials: int) -> pd.Series:
    duration = pd.Series(np.nan, index=range(n_trials), dtype=float)
    if not flat.empty and {"slide_index", "system_time"}.issubset(flat.columns):
        for idx, group in flat.dropna(subset=["system_time"]).groupby("slide_index"):
            if len(group) >= 2 and int(idx) in duration.index:
                duration.loc[int(idx)] = float(group["system_time"].max() - group["system_time"].min())
    return duration


def run_fixations(flat: pd.DataFrame, output_dir: Path, params: dict) -> pd.DataFrame:
    required = {"set_name", "slide_index", "avg_gaze_x", "avg_gaze_y", "system_time"}
    if flat.empty or not required.issubset(flat.columns):
        return pd.DataFrame(columns=["set_name","slide_index","fix_start","fix_end","duration","x_mean","y_mean","dispersion"])
    clean = flat.dropna(subset=["avg_gaze_x","avg_gaze_y","system_time"]).copy()
    if clean.empty:
        return pd.DataFrame(columns=["set_name","slide_index","fix_start","fix_end","duration","x_mean","y_mean","dispersion"])
    analyzer = FixationAnalyzer(output_dir, **params)
    return analyzer.analyze(clean)


def run_scanpaths(fixations: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    if fixations.empty:
        return pd.DataFrame(columns=["set_name","slide_index","distance"])
    return ScanpathsAnalyzer(output_dir).analyze(fixations, per="slide")


def summarize_fixations(fixations: pd.DataFrame, n_trials: int) -> pd.DataFrame:
    base = pd.DataFrame({"slide_index": range(n_trials)})
    if fixations.empty:
        return base.assign(fixation_count=0, total_fixation_duration=0.0, mean_fixation_duration=np.nan)
    agg = (fixations.groupby("slide_index")
           .agg(fixation_count=("duration","size"),
                total_fixation_duration=("duration","sum"),
                mean_fixation_duration=("duration","mean"))
           .reset_index())
    return base.merge(agg, on="slide_index", how="left").fillna({"fixation_count":0,"total_fixation_duration":0.0})


def summarize_scanpaths(scanpaths: pd.DataFrame, n_trials: int) -> pd.DataFrame:
    base = pd.DataFrame({"slide_index": range(n_trials)})
    if scanpaths.empty:
        return base.assign(scanpath_transition_count=0, scanpath_distance=0.0)
    agg = (scanpaths.groupby("slide_index")
           .agg(scanpath_transition_count=("distance","size"), scanpath_distance=("distance","sum"))
           .reset_index())
    return base.merge(agg, on="slide_index", how="left").fillna({"scanpath_transition_count":0,"scanpath_distance":0.0})

DEMO_ROOT = find_demo_root()
print(f"Demo repository: {DEMO_ROOT}")

from tobii_pytracker.analyze import HeatmapAnalyzer, FocusMapAnalyzer, SaccadeAnalyzer, EntropyAnalyzer, ClusterAnalyzer, BBoxAttentionAnalyzer

In [ ]:
example_dir=DEMO_ROOT/"examples"/"image_semantic_demo"
config_path = next(example_dir.glob("config*.yaml"))
loader, SESSION, raw, flat = prepare_session(config_path, SELECTED_SESSION)
manifest=pd.read_csv(example_dir/"stimuli_manifest.csv")
expected=dict(zip(manifest["filename"].astype(str),manifest["class"].astype(str).str.casefold()))
raw["filename"]=raw["input_data"].map(lambda x:Path(str(x).replace("\\","/")).name)
raw["semantic_class"]=raw["filename"].map(expected)
raw["response"]=raw["user_classification"].astype(str).str.casefold()
raw["correct"]=raw["response"]==raw["semantic_class"]
analysis_dir=loader.output_root/SESSION/"analysis_image_semantic_demo_v2"; analysis_dir.mkdir(exist_ok=True)
print(f"Session: {SESSION}; trials={len(raw)}; gaze samples={len(flat)}")

## 3. Reproducibility record and design integrity

Before analyzing gaze, the notebook verifies the expected 12-image 4/4/4 category balance and checks the constrained mixed order recorded in the session. This ensures that category summaries are not based on an incomplete or accidentally blocked sequence.

File hashes and environment information provide provenance for the exact session and stimulus manifest used in the analysis.


In [ ]:
import hashlib
import platform
from importlib.metadata import PackageNotFoundError, version as package_version


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def installed_version(distribution: str) -> str:
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return "package-metadata-unavailable"


def provenance_table(session: str, data_csv: Path, dataset_path: Path, analysis_label: str) -> pd.DataFrame:
    record = {
        "analysis_label": analysis_label,
        "session": str(session),
        "data_csv_sha256": sha256_file(data_csv),
        "dataset_sha256": sha256_file(dataset_path),
        "python": platform.python_version(),
        "tobii_pytracker": installed_version("tobii-pytracker"),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    }
    return pd.DataFrame([record])


def save_json(path: Path, payload: dict):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")

dataset_path=example_dir/"stimuli_manifest.csv"; data_csv=loader.output_root/SESSION/"data.csv"
provenance=provenance_table(SESSION,data_csv,dataset_path,"image_semantic_demo_v2")
sequence=raw["semantic_class"].astype(str).tolist(); expected_classes={"biological","object","scene"}
adjacent_mixed=all(sequence[i]!=sequence[i-1] for i in range(1,len(sequence)))
triplets_balanced=(len(sequence)==12 and all(set(sequence[i:i+3])==expected_classes for i in range(0,12,3)))
design_checks=pd.DataFrame([
    {"check":"12 recorded trials","passed":len(raw)==12,"observed":len(raw),"expected":12},
    {"check":"4/4/4 category balance","passed":raw["semantic_class"].value_counts().to_dict()=={"biological":4,"object":4,"scene":4},"observed":str(raw["semantic_class"].value_counts().to_dict()),"expected":"4 each"},
    {"check":"no adjacent category repeats","passed":adjacent_mixed,"observed":adjacent_mixed,"expected":True},
    {"check":"each 3-trial cycle contains all categories","passed":triplets_balanced,"observed":triplets_balanced,"expected":True},
    {"check":"responses restricted to task classes","passed":set(raw["response"].dropna()).issubset(expected_classes),"observed":str(sorted(set(raw["response"].dropna()))),"expected":str(sorted(expected_classes))},
])
display(provenance); display(design_checks)
if not bool(design_checks["passed"].all()): warnings.warn("The selected session does not fully match the canonical mixed 12-image design.")

## 4. Data-quality audit

Behavioral classification and gaze availability are reported separately. A trial without usable gaze remains valid for accuracy analysis but contributes no fixation, saccade, entropy, heatmap, or AOI-attention measure.

Because the task is forced choice with no uncertainty button, response completeness should be nearly perfect; unexpected missing responses should be investigated as a collection problem.


In [ ]:

gaze_counts=trial_gaze_counts(flat,len(raw)); observed_duration=trial_observed_duration(flat,len(raw))
quality=raw[["slide_index","filename","semantic_class","response","correct"]].copy()
quality["gaze_samples"]=quality["slide_index"].map(gaze_counts); quality["usable_gaze"]=quality["gaze_samples"]>0
quality["observed_gaze_duration_s"]=quality["slide_index"].map(observed_duration)
quality_overview=pd.DataFrame([
    {"metric":"recorded_trials","value":len(quality)},
    {"metric":"trials_with_usable_gaze","value":int(quality["usable_gaze"].sum())},
    {"metric":"trials_without_usable_gaze","value":int((~quality["usable_gaze"]).sum())},
    {"metric":"behavioral_accuracy","value":float(quality["correct"].mean())},
])
display(quality_overview); display(quality)
if (~quality["usable_gaze"]).any(): warnings.warn("Category responses are retained; eye-movement metrics are missing for trials without gaze.")


## 5. Built-in fixation, saccade, scanpath, and entropy analyzers

This section demonstrates the core public analyzers rather than reimplementing them locally. The measures provide complementary views of spatial exploration:

- **fixations** describe stable viewing events;
- **saccades** describe rapid transitions between viewed locations;
- **scanpaths** summarize fixation-to-fixation movement;
- **entropy / convex hull** describe how concentrated or dispersed gaze is in screen space.

None of these quantities should be interpreted alone as cognitive load or difficulty. Their scientific value comes from comparison with category, accuracy, and stimulus identity.

> **tobii-pytracker support:** This section directly uses `FixationAnalyzer`, `SaccadeAnalyzer`, `ScanpathsAnalyzer`, and `EntropyAnalyzer` for the corresponding event and spatial summaries.


In [ ]:
clean_flat=flat.dropna(subset=["avg_gaze_x","avg_gaze_y","system_time"]).copy() if not flat.empty else pd.DataFrame()
fixations=run_fixations(flat,analysis_dir,FIXATION_PARAMS)
scanpaths=run_scanpaths(fixations,analysis_dir)
if clean_flat.empty:
    saccades=pd.DataFrame(); spatial_entropy=pd.DataFrame()
else:
    saccades=SaccadeAnalyzer(analysis_dir,**SACCADE_PARAMS).analyze(clean_flat)
    spatial_entropy=EntropyAnalyzer(analysis_dir).analyze(clean_flat,per="slide",bins=SPATIAL_ENTROPY_BINS)
fix_summary=summarize_fixations(fixations,len(raw)); scan_summary=summarize_scanpaths(scanpaths,len(raw))
if saccades.empty:
    saccade_summary=pd.DataFrame({"slide_index":range(len(raw)),"saccade_count":0,"mean_saccade_amplitude":np.nan,"total_saccade_amplitude":0.0})
else:
    saccade_summary=(saccades.groupby("slide_index").agg(saccade_count=("amplitude","size"),mean_saccade_amplitude=("amplitude","mean"),total_saccade_amplitude=("amplitude","sum")).reset_index())

## 6. Optional fixation-parameter sensitivity

Use this diagnostic to verify that category-level fixation patterns are not an artifact of a single dispersion threshold.

In [ ]:
sensitivity = pd.DataFrame()
if RUN_FIXATION_SENSITIVITY:
    rows = []
    for threshold in SENSITIVITY_DISPERSION_THRESHOLDS:
        params = dict(FIXATION_PARAMS)
        params["dispersion_threshold"] = float(threshold)
        detected = run_fixations(flat, analysis_dir / "sensitivity", params)
        rows.append({
            "dispersion_threshold": float(threshold),
            "fixation_count": int(len(detected)),
            "mean_fixation_duration": float(detected["duration"].mean()) if not detected.empty else np.nan,
        })
    sensitivity = pd.DataFrame(rows)
    display(sensitivity)
else:
    print("Sensitivity analysis is disabled. Set RUN_FIXATION_SENSITIVITY=True to compare fixation thresholds.")

## 7. Built-in 3×3 image-bbox attention analysis

The 3×3 grid is the experiment's predefined spatial AOI system. `BBoxAttentionAnalyzer` can consume these `image_bboxes` directly, so this section provides the clearest demonstration of the library's AOI functionality.

Raw-gaze attention and fixation-based attention answer slightly different questions: raw samples describe the distribution of the gaze stream, while fixation-based scoring emphasizes stable viewing episodes. Grid-cell coverage and central dwell are therefore reported separately from heatmaps and entropy.

> **tobii-pytracker support:** The 3×3 AOI scoring is handled directly by the public `BBoxAttentionAnalyzer` using the `image_bboxes` stored with each trial.


In [ ]:
if clean_flat.empty:
    bbox_gaze=pd.DataFrame(); bbox_fix=pd.DataFrame(); bbox_eval=pd.DataFrame()
else:
    bbox_analyzer=BBoxAttentionAnalyzer(analysis_dir)
    bbox_gaze=bbox_analyzer.analyze(raw,clean_flat,use_fixations=False)
    bbox_eval=bbox_analyzer.evaluate(bbox_gaze)
    bbox_fix=bbox_analyzer.analyze(raw,fixations,use_fixations=True) if not fixations.empty else pd.DataFrame()

def add_grid_labels(df):
    if df.empty: return df.copy()
    out=df.copy(); labels=[]
    for (set_name,slide),g in out.groupby(["set_name","slide_index"]):
        xs=sorted(g["cx"].unique()); ys=sorted(g["cy"].unique(), reverse=True)
        xnames={x:n for x,n in zip(xs,["left","center","right"])}
        ynames={y:n for y,n in zip(ys,["top","middle","bottom"])}
        for idx,row in g.iterrows(): labels.append((idx,f"{ynames.get(row.cy,'?')}-{xnames.get(row.cx,'?')}"))
    mapping=dict(labels); out["grid_cell"]=[mapping[i] for i in out.index]; return out
bbox_gaze=add_grid_labels(bbox_gaze); bbox_fix=add_grid_labels(bbox_fix)
display(bbox_eval.head())

## 8. Trial-level metrics and category comparison

Category summaries correspond to H1–H3 in the demo documentation, but the notebook also retains per-image metrics for H4. This is essential because each category contains only four visually heterogeneous exemplars.

Interpret category-level differences only after checking the stimulus-level table. A large effect driven by a single image is an item effect, not strong evidence for a general semantic-category phenomenon.


In [ ]:

trial_metrics=(raw[["slide_index","filename","semantic_class","response","correct"]]
 .merge(quality[["slide_index","gaze_samples","usable_gaze","observed_gaze_duration_s"]],on="slide_index")
 .merge(fix_summary,on="slide_index").merge(scan_summary,on="slide_index").merge(saccade_summary,on="slide_index",how="left"))
if not spatial_entropy.empty:
    trial_metrics=trial_metrics.merge(spatial_entropy[["slide_index","entropy","convex_hull_area"]],on="slide_index",how="left")
if not bbox_eval.empty:
    trial_metrics=trial_metrics.merge(bbox_eval[["slide_index","attended_bboxes","attended_bbox_ratio","coverage_by_bboxes"]],on="slide_index",how="left")

category_grid=pd.DataFrame(); per_image_grid=pd.DataFrame()
if not bbox_fix.empty:
    labeled=bbox_fix.merge(raw[["slide_index","filename","semantic_class"]],on="slide_index",how="left")
    per_image_grid=(labeled.groupby(["slide_index","filename","semantic_class","grid_cell"],as_index=False)["dwell_time"].sum())
    totals=per_image_grid.groupby("slide_index")["dwell_time"].transform("sum")
    per_image_grid["dwell_share"]=np.where(totals>0,per_image_grid["dwell_time"]/totals,np.nan)
    category_grid=(per_image_grid.groupby(["semantic_class","grid_cell"],as_index=False)["dwell_time"].sum())
    category_totals=category_grid.groupby("semantic_class")["dwell_time"].transform("sum")
    category_grid["dwell_share"]=np.where(category_totals>0,category_grid["dwell_time"]/category_totals,np.nan)
    central=(per_image_grid[per_image_grid["grid_cell"].eq("middle-center")][["slide_index","dwell_share"]].rename(columns={"dwell_share":"central_fixation_dwell_share"}))
    trial_metrics=trial_metrics.merge(central,on="slide_index",how="left")

eye_cols=["fixation_count","total_fixation_duration","mean_fixation_duration","scanpath_transition_count","scanpath_distance","saccade_count","mean_saccade_amplitude","total_saccade_amplitude","entropy","convex_hull_area","attended_bboxes","attended_bbox_ratio","coverage_by_bboxes","central_fixation_dwell_share"]
existing=[c for c in eye_cols if c in trial_metrics.columns]
trial_metrics.loc[~trial_metrics["usable_gaze"],existing]=np.nan

display(trial_metrics)
category_summary=(trial_metrics.groupby("semantic_class")
 .agg(n_trials=("slide_index","size"),accuracy=("correct","mean"),usable_gaze_rate=("usable_gaze","mean"),mean_observed_gaze_duration_s=("observed_gaze_duration_s","mean"),
      mean_fixation_count=("fixation_count","mean"),mean_fixation_duration=("mean_fixation_duration","mean"),mean_saccade_count=("saccade_count","mean"),
      mean_saccade_amplitude=("mean_saccade_amplitude","mean"),mean_scanpath_distance=("scanpath_distance","mean"),mean_spatial_entropy=("entropy","mean"),
      mean_convex_hull_area=("convex_hull_area","mean"),mean_grid_cells_visited=("attended_bboxes","mean"),mean_central_dwell_share=("central_fixation_dwell_share","mean")).reset_index())
display(category_summary)

confusion=pd.crosstab(trial_metrics["semantic_class"],trial_metrics["response"],dropna=False)
per_image_diagnostics=trial_metrics.sort_values(["correct","fixation_count","entropy"],ascending=[True,False,False],na_position="last")
display(confusion); display(per_image_diagnostics)


## 9. Representative-trial visualizations using built-in analyzers

Representative plots connect numeric summaries back to the visible stimulus. Heatmaps show gaze density, focus maps emphasize highly viewed regions, and fixation/saccade overlays reveal event sequences.

These figures are diagnostic and illustrative. They are particularly useful for detecting geometry errors or atypical trials, but visual appearance alone should not replace the quantitative trial-level summaries.

> **tobii-pytracker support:** Gaze, heatmap, focus-map, entropy, fixation, saccade, scanpath, and bbox-attention visualizations in this section use plotting helpers supplied by `tobii-pytracker`.


In [ ]:

representative_pool=quality[quality["usable_gaze"]].copy()
if REPRESENTATIVE_CATEGORY is not None:
    representative_pool=representative_pool[representative_pool["semantic_class"].eq(str(REPRESENTATIVE_CATEGORY).casefold())]
if not representative_pool.empty:
    representative=int(representative_pool.sort_values("gaze_samples",ascending=False).iloc[0]["slide_index"])
    slide_meta=loader.get_slide_data(SESSION,representative,flatten=False); screenshot=Path(slide_meta["screenshot_path"])
    slide_flat=clean_flat[clean_flat["slide_index"]==representative]
    print(f"Representative trial: {representative} — {raw.loc[representative,'filename']}")
    if not slide_flat.empty and screenshot.exists():
        loader.plot_gaze(SESSION,representative,gradient=True,show=True)
        HeatmapAnalyzer(analysis_dir).plot_analysis(slide_flat,screenshot,title="Gaze heatmap",show=True)
        FocusMapAnalyzer(analysis_dir).plot_analysis(slide_flat,screenshot,title="Focus map",show=True)
        EntropyAnalyzer(analysis_dir).plot_analysis(slide_flat,screenshot,title="Entropy / convex-hull diagnostic",show=True)
        if not fixations[fixations.slide_index==representative].empty:
            FixationAnalyzer(analysis_dir,**FIXATION_PARAMS).plot_analysis(fixations,screenshot,set_name=SESSION,slide_index=representative,show=True)
        if not saccades.empty and not saccades[saccades.slide_index==representative].empty:
            SaccadeAnalyzer(analysis_dir,**SACCADE_PARAMS).plot_analysis(saccades,screenshot,set_name=SESSION,slide_index=representative,show=True)
        if not scanpaths.empty and not scanpaths[scanpaths.slide_index==representative].empty:
            ScanpathsAnalyzer(analysis_dir).plot_analysis(scanpaths,screenshot,set_name=SESSION,slide_index=representative,show=True)
        if not bbox_gaze.empty:
            BBoxAttentionAnalyzer(analysis_dir).plot_analysis(bbox_gaze,clean_flat,screenshot,set_name=SESSION,slide_index=representative,top_k=9,show=True)
else:
    representative=None; slide_flat=pd.DataFrame(); screenshot=None
    print("No representative trial with usable gaze is available.")


## 10. Exploratory clustering

Clustering asks whether gaze samples form data-driven spatial concentrations within a trial. DBSCAN is used here only as an exploratory demonstration of the built-in `ClusterAnalyzer`.

Clusters are **not semantic AOIs** and should not be labeled as objects without independent annotation. Results can change materially with `eps`, sample density, and gaze-quality characteristics, so this section should remain secondary to the predefined 3×3 grid analysis.

> **tobii-pytracker support:** The exploratory spatial clusters are generated with the library's `ClusterAnalyzer`; interpretation remains exploratory rather than confirmatory.


In [ ]:

clustered=pd.DataFrame(); cluster_summary=pd.DataFrame()
if representative is not None and not slide_flat.empty:
    try:
        cluster_analyzer=ClusterAnalyzer(analysis_dir,eps=CLUSTER_EPS,min_samples=CLUSTER_MIN_SAMPLES)
        clustered=cluster_analyzer.analyze(slide_flat)
        valid_clusters=clustered.loc[clustered["cluster"].ge(0),"cluster"] if "cluster" in clustered else pd.Series(dtype=float)
        cluster_summary=pd.DataFrame([{"slide_index":representative,"cluster_count":int(valid_clusters.nunique()),"noise_fraction":float((clustered["cluster"]<0).mean()) if "cluster" in clustered else np.nan}])
        display(cluster_summary)
        if screenshot is not None and Path(screenshot).exists():
            cluster_analyzer.plot_analysis(clustered,screenshot,set_name=SESSION,slide_index=representative,title="Exploratory gaze clusters",show=True)
    except Exception as exc:
        warnings.warn(f"Exploratory clustering skipped: {exc}")


## 11. Visual category summaries

Category-level matrices and plots summarize how attention is distributed across the 3×3 grid. They are designed to make H1 and H2 interpretable: broad SCENE exploration, for example, should appear as attention distributed across more cells rather than being inferred only from a single entropy value.

Always inspect the corresponding per-image diagnostics because averaging four heterogeneous stimuli can conceal meaningful exemplar differences.


In [ ]:

fig,axes=plt.subplots(2,2,figsize=(12,9))
category_summary.set_index("semantic_class")["accuracy"].plot(kind="bar",ax=axes[0,0]); axes[0,0].set_ylim(0,1); axes[0,0].set_title("Behavioral accuracy")
trial_metrics.boxplot(column="fixation_count",by="semantic_class",ax=axes[0,1]); axes[0,1].set_title("Fixation count")
trial_metrics.boxplot(column="entropy",by="semantic_class",ax=axes[1,0]); axes[1,0].set_title("Spatial entropy")
trial_metrics.boxplot(column="attended_bboxes",by="semantic_class",ax=axes[1,1]); axes[1,1].set_title("Grid cells attended")
plt.suptitle(""); plt.tight_layout(); plt.show()

if not category_grid.empty:
    order=["top-left","top-center","top-right","middle-left","middle-center","middle-right","bottom-left","bottom-center","bottom-right"]
    classes=sorted(category_grid["semantic_class"].dropna().unique())
    fig,axes=plt.subplots(1,len(classes),figsize=(5*len(classes),4),squeeze=False)
    for ax,category in zip(axes.flat,classes):
        values=category_grid[category_grid["semantic_class"].eq(category)].set_index("grid_cell")["dwell_share"].reindex(order).fillna(0).to_numpy().reshape(3,3)
        im=ax.imshow(values,vmin=0,vmax=max(0.01,float(values.max())))
        ax.set_title(f"{category}: fixation-dwell share"); ax.set_xticks(range(3),["left","center","right"]); ax.set_yticks(range(3),["top","middle","bottom"])
        for r in range(3):
            for c in range(3): ax.text(c,r,f"{values[r,c]:.2f}",ha="center",va="center")
    plt.tight_layout(); plt.show()

fig,ax=plt.subplots(figsize=(10,4))
plot_df=per_image_diagnostics.dropna(subset=["fixation_count"]).sort_values("fixation_count",ascending=False)
if not plot_df.empty:
    ax.bar(plot_df["filename"],plot_df["fixation_count"]); ax.tick_params(axis="x",rotation=60); ax.set_ylabel("Fixation count"); ax.set_title("Stimulus-level heterogeneity")
    plt.tight_layout(); plt.show()


## 12. Export derived tables

The notebook exports derived trial-, category-, grid-, and stimulus-level results while leaving `data.csv` unchanged. Provenance and analysis parameters should be retained with the exports so visual summaries can be regenerated from the same session.

For group analysis, preserve image identity and participant identity. Semantic category alone is insufficient because stimulus heterogeneity is an explicit limitation of this demo.


In [ ]:

trial_metrics.to_csv(analysis_dir/"trial_metrics.csv",index=False); category_summary.to_csv(analysis_dir/"category_summary.csv",index=False)
confusion.to_csv(analysis_dir/"response_confusion.csv"); per_image_diagnostics.to_csv(analysis_dir/"per_image_diagnostics.csv",index=False)
quality.to_csv(analysis_dir/"data_quality_trials.csv",index=False); quality_overview.to_csv(analysis_dir/"data_quality_summary.csv",index=False)
fixations.to_csv(analysis_dir/"fixations.csv",index=False); saccades.to_csv(analysis_dir/"saccades.csv",index=False); scanpaths.to_csv(analysis_dir/"scanpaths.csv",index=False)
bbox_gaze.to_csv(analysis_dir/"bbox_attention_gaze.csv",index=False); bbox_fix.to_csv(analysis_dir/"bbox_attention_fixations.csv",index=False)
category_grid.to_csv(analysis_dir/"category_grid_attention.csv",index=False); per_image_grid.to_csv(analysis_dir/"per_image_grid_attention.csv",index=False)
cluster_summary.to_csv(analysis_dir/"representative_cluster_summary.csv",index=False)
if not sensitivity.empty: sensitivity.to_csv(analysis_dir/"fixation_sensitivity.csv",index=False)
provenance.to_csv(analysis_dir/"provenance.csv",index=False); design_checks.to_csv(analysis_dir/"design_checks.csv",index=False)
save_json(analysis_dir/"analysis_parameters.json",{"fixation":FIXATION_PARAMS,"saccade":SACCADE_PARAMS,"entropy_bins":SPATIAL_ENTROPY_BINS,"cluster_eps":CLUSTER_EPS,"cluster_min_samples":CLUSTER_MIN_SAMPLES,"sensitivity_thresholds":SENSITIVITY_DISPERSION_THRESHOLDS})
print(analysis_dir)


## 13. Scientific interpretation and limitations

The demo documentation frames the task as a study of **spatial evidence sampling during semantic categorization**. Read the results in that order: behavior first, then spatial exploration, then optional exploratory structure.

- A broad scanpath, large convex hull, or high entropy describes distributed gaze; it does not automatically imply poor performance or high cognitive load.
- The 3×3 grid is a coarse spatial partition, not a semantic object annotation. Grid-cell attention should therefore be described spatially.
- The built-in `BBoxAttentionAnalyzer` is used directly because `image_bboxes` match its public API.
- Heatmaps and focus maps visualize density but do not identify why a region was informative.
- Stimulus-level diagnostics are essential because four heterogeneous exemplars per category cannot support strong category-general conclusions.
- Missing gaze is excluded from eye-movement metrics but does not invalidate the behavioral response.
- DBSCAN clustering is exploratory and parameter-sensitive; clusters are not semantic AOIs.
- Confirmatory category effects require more participants, more controlled stimuli, preregistered parameters, and participant/stimulus-aware models.

**Documentation link:** `docs/basic_examples/image_semantic_demo.md`.
